> **구분: 수업 후 복습·자습**

# 01. 작은 모델의 훈련 구조 — 20분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 현재 실습의 입력·저장·실행 범위

열 파일은 `/workspace/04.LLM/01_training_basics.ipynb`입니다. 첫 환경 셀은 `ROOT`, `COURSE`, `DATA`, `OUTPUTS`, `check`, `load_op`, `run_training`을 **이 커널에 다시 정의** 합니다. Python 경로와 PyTorch·CUDA 출력에서 CUDA가 `True`인지 확인하고 같은 Jetson의 다른 사용자와 GPU 실행 순서를 맞춥니다. 이 숫자 모델 훈련에는 Ollama 서버가 필요하지 않습니다.

입력은 `/workspace/Data/datasets/04.LLM/digits.json`과 `.sha256`입니다. 이 데이터 640개를 train 512개·validation 128개로 나누며, PAD/BOS/EOS=0/1/2, 숫자 0~9의 token ID=3~12입니다. 파일이 없으면 배포 데이터가 포함된 개인 자료 폴더인지 확인합니다. 이전 훈련 checkpoint 없이 새 훈련을 시작합니다.

`exercises/*.py`는 Jupyter 파일 편집기에서 수정·저장하세요. 노트북의 파일 출력 셀은 읽기 전용이고 `check(..., 'exercise')`는 저장한 최신 파일을 읽습니다. 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`·`loss.csv`, 자신의 가중치는 `/workspace/Data/checkpoints/users/04.LLM/`에 남습니다. `run_training`은 새 결과 폴더를 만들고 반환한 `report['checkpoint']`는 `Data` 기준 상대 경로입니다. 다른 노트북의 `report` 변수는 이어지지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

이 선택 자습은 모델을 호출하기 전에 입력이 어떤 숫자로 바뀌고 가중치가 어떻게 수정되는지 확인합니다. token의 정수 번호 자체에 크기 의미가 있는 것은 아니며 embedding은 그 번호에 연결된 학습 가능한 벡터입니다.

### 전체 과정에서 내 코드가 하는 일

```text
입력 기호
  ↓
token ID
  ↓
embedding·예측 logits
  ↓
정답과 loss
  ↓
gradient
  ↓
optimizer step
```

각 tensor 축과 타입을 확인하고 제공 TODO를 구현합니다. train/eval은 모델 동작 모드를, no_grad는 gradient 기록 여부를 다루므로 같은 기능으로 이해하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** eval을 호출하면 가중치가 원래대로 돌아가나요?

**설명:** 아닙니다. eval은 추론 시 동작 모드 설정이며 저장 가중치를 복구하는 기능이 아닙니다. 새로운 가중치를 읽으려면 체크포인트 로드를 따로 수행합니다. gradient 초기화·계산·갱신의 순서도 직접 추적합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| loss 검사 실패·학습이 진행되지 않음 | loss TODO의 shape·평균 방식·padding 처리와 check 메시지를 비교합니다. | 해당 check(..., exercise) 통과 |
| `ModuleNotFoundError: common` / 다른 함수가 로딩됨 | 작업 경로와 커널을 확인합니다. 다른 과목 common이 남으면 저장 후 해당 커널을 재시작하고 초기 셀부터 실행합니다. | 현재 04.LLM 모듈 import 성공 |
| 훈련 timeout·OOM·loss NaN | 직전 stdout/stderr와 설정을 보존합니다. 작은 quick profile부터 확인하고 중복 실행·입력 유한성·device를 점검합니다. | 작은 훈련 완료, 유한 loss와 metrics 저장 |

**예시에 없는 오류를 확인할 곳:** [LLM README](README.md), [contracts 검사](common/contracts.py), exercises/solutions의 해당 연산. 실행이 출력한 output 경로의 metrics.json과 터미널 오류를 확인하세요. [PyTorch 이슈](https://github.com/pytorch/pytorch/issues)는 버전과 최소 tensor 예제로 검색합니다.


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

본수업 훈련 120분은 [00_training_core.ipynb](00_training_core.ipynb)에서 진행합니다. 이 노트북 전체는 강사 선택 시연·자습이며 여기에 적힌 시간은 별도 자습 예상 시간입니다. 다섯 모델 구현 완료는 본수업 통과 조건이 아닙니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 시간·목표

환경·핵심 지도 4분 → loss TODO 6분 → GPU 역전파 확인 7분 → 결과 점검 3분. 목표는 logits, target, gradient와 훈련/검증의 차이를 설명하는 것입니다.

### 자습 진행 경로

1. 환경 셀을 실행합니다.
2. 바로 아래 **20분 수업 핵심 지도** 를 읽습니다. 번호가 붙은 1–7절은 수업 후 상세 복습 자료입니다.
3. `자습 구현 TODO — loss`에서 파일을 수정하고 학습자 검증을 통과합니다.
4. GPU의 loss/backward/step 셀을 실행하고 마지막 완료 질문에 답합니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import importlib, json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.contracts import check
from common.loading import load_op
import torch
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '훈련은 Jetson GPU에서 수행합니다. 강사에게 환경 점검을 요청하세요.'
def run_training(kind, variant='exercise'):
    before = {p.resolve() for p in OUTPUTS.glob('*/metrics.json')}
    command = [sys.executable, str(COURSE / 'scripts/train.py'), '--model', kind, '--variant', variant]
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    created = [p for p in OUTPUTS.glob('*/metrics.json') if p.resolve() not in before]
    assert len(created) == 1, f'새 metrics.json이 정확히 1개여야 합니다: {created}'
    metrics_path = created[0]
    report = json.loads(metrics_path.read_text())
    assert report['kind'] == kind and report['variant'] == variant and report['profile'] == 'quick'
    report['_metrics_path'] = str(metrics_path.relative_to(DATA))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    if kind in ('rnn','lstm'):
        print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    else:
        print('자유 생성 token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('자유 생성 시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    print('실행 경로:', report['run_path'])
    return report


> **구분: 수업 후 복습·자습**

## 20분 수업 핵심 지도

- 모델은 입력을 출력으로 바꾸며, 훈련은 loss가 작아지는 방향으로 파라미터를 바꿉니다.
- token ID `[B,T]` → embedding `[B,T,H]` → logits `[B,T,V]` 순서로 shape가 변합니다.
  - **약어:** B(batch size)는 한 번에 처리하는 샘플 수, T(sequence length)는 샘플의 토큰 수, H(hidden size)는 내부 표현 벡터 크기, V(vocabulary size)는 토큰 종류 수입니다.
- logits는 확률 전 점수이고, target은 정답 token ID `[B,T]`입니다.
- `CrossEntropyLoss`에는 logits `[B×T,V]`와 target `[B×T]`를 넣습니다. softmax를 먼저 적용하지 않습니다.
- 한 step은 `zero_grad → forward → loss → backward → optimizer.step`입니다. `backward`는 gradient 계산, `step`은 파라미터 변경입니다.
- 훈련 데이터는 파라미터를 바꾸는 데 쓰고, 검증 데이터는 바꾸지 않은 새 예제에서 상태를 확인하는 데 씁니다.

이 요약을 확인한 뒤 곧바로 loss TODO로 이동해도 됩니다. 아래 1–7절은 같은 개념을 비유와 예제로 풀어 쓴 복습 자료입니다.


> **구분: 수업 후 복습·자습**

## 1. AI, 머신러닝, 딥러닝, 언어 모델의 관계

먼저 비슷해 보이는 용어의 범위를 구분합니다.

- **인공지능(AI)** : 사람이 지능이 필요하다고 느끼는 문제를 컴퓨터로 해결하려는 넓은 분야입니다. 규칙을 사람이 직접 쓰는 프로그램도 AI에 포함될 수 있습니다.
- **머신러닝(ML)** : 규칙을 모두 손으로 쓰는 대신, 데이터에서 규칙에 해당하는 **파라미터(parameter)** 를 찾게 하는 방법입니다.
- **딥러닝(DL)** : 여러 층의 신경망을 사용하는 머신러닝의 한 갈래입니다. 이 수업의 RNN, LSTM, Transformer가 여기에 속합니다.
- **언어 모델(language model)** : 앞의 token들을 보고 다음 token이 얼마나 그럴듯한지 점수를 내는 모델입니다.
- **대규모 언어 모델(LLM)** : 매우 많은 데이터와 파라미터로 사전학습한 언어 모델입니다. '대규모'의 경계는 고정된 숫자가 아니라 시대와 문맥에 따라 달라집니다.

**모델** 은 입력을 출력으로 바꾸는 함수이고, **파라미터** 는 훈련 중 바뀌는 숫자들입니다. **훈련(training)** 은 데이터에 맞게 파라미터를 조절하는 과정이고, **추론(inference)** 은 조절을 끝낸 파라미터로 새 입력의 출력을 계산하는 과정입니다.

이 다섯 개의 작은 모델(RNN, LSTM, Seq2Seq, Attention Seq2Seq, Transformer)을 실습하는 목적은 거대한 Gemma를 재현하는 것이 아닙니다. 작은 숫자 데이터로도 token, embedding, loss, gradient, attention과 같은 핵심 계산을 직접 볼 수 있기 때문입니다. 5–6번 노트북에서는 별도 Ollama 서버에 있는 **이미 사전학습된 Gemma** 를 추론에 사용합니다.


> **구분: 수업 후 복습·자습**

## 2. 데이터, 훈련 집합, 검증 집합

머신러닝 모델은 예제에서 규칙을 찾습니다. 이때 모델이 파라미터를 바꾸는 데 직접 사용하는 예제를 **훈련 집합(train set)** , 훈련에 쓰지 않고 중간 성능을 확인하는 예제를 **검증 집합(validation set)** 이라고 합니다.

시험 문제를 미리 외우면 실력을 공정하게 평가할 수 없는 것처럼, 검증 데이터를 훈련에 섞으면 새 입력을 처리하는 능력을 과대평가하게 됩니다. 이 자료는 서로 다른 숫자 시퀀스 640개를 훈련 512개와 검증 128개로 나눕니다. 실제 프로젝트에서는 마지막 한 번의 평가용 **테스트 집합(test set)** 도 별도로 두지만, 짧은 교육 실습에서는 훈련과 검증의 차이에 집중합니다.

여기서 검증 성능이 좋아도 '자연어를 이해했다'는 뜻은 아닙니다. 길이 5, 숫자 0–9라는 매우 작은 규칙 안에서 아직 보지 않은 조합을 처리했다는 뜻입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from common.paths import DATASET
dataset = json.loads(DATASET.read_text())
print('train:', len(dataset['train']), 'validation:', len(dataset['validation']))
print('첫 원시 token:', dataset['train'][0])
print('사람이 읽는 숫자:', [v-3 for v in dataset['train'][0]])
print('두 집합에 같은 시퀀스가 있는가?', bool(set(map(tuple, dataset['train'])) & set(map(tuple, dataset['validation']))))

> **구분: 수업 후 복습·자습**

## 3. token과 vocabulary

신경망은 문자열을 바로 계산하지 못하므로 먼저 작은 단위로 나누고 정수 번호를 붙입니다.

- **token** : 모델이 한 단위로 처리하는 조각입니다. 자연어에서는 한 글자, 단어, 또는 단어의 일부일 수 있습니다.
- **tokenization** : 텍스트를 token으로 나누고 번호로 바꾸는 과정입니다.
- **vocabulary(어휘 집합)** : 모델이 구분하는 모든 token과 번호의 목록입니다. 크기를 `V`로 씁니다.

이 실습의 vocabulary 크기는 13입니다. 숫자 0–9를 token 번호 3–12에 대응시키고, 다음 특수 token 세 개를 둡니다.

| 번호 | 이름 | 역할 |
|---:|---|---|
| 0 | PAD | 길이가 다른 입력을 한 batch로 묶을 때 빈 자리를 채움 |
| 1 | BOS | 출력 시퀀스의 시작을 알림(Beginning Of Sequence) |
| 2 | EOS | 출력 시퀀스가 끝났음을 알림(End Of Sequence) |

token 번호는 크기나 의미의 순서를 나타내지 않는 **범주 ID** 입니다. token 12가 token 3보다 아홉 배 크거나 더 중요하다는 뜻이 아닙니다. 이번 입력 길이는 모두 5라 PAD를 실제 계산에 쓰지는 않지만, 이후 자연어 배치를 이해하기 위해 vocabulary에 포함합니다.


> **구분: 수업 후 복습·자습**

## 4. 텐서와 shape를 읽는 법

**tensor(텐서)** 는 같은 자료형의 숫자를 여러 축으로 정리한 배열입니다. 스칼라는 축이 0개, 벡터는 1개, 행렬은 2개인 텐서라고 볼 수 있습니다. PyTorch의 거의 모든 계산은 텐서를 입력받습니다.

이 자료에서 자주 보는 기호는 다음과 같습니다.

| 기호 | 뜻 | 이번 실습 예 |
|---|---|---:|
| `B` | batch size: 동시에 처리하는 샘플 수 | 2 또는 32 |
| `T` | sequence length: 샘플 하나의 token 위치 수 | 입력 5, 역순 정답 6 |
| `H` | hidden size: token을 표현하는 내부 실수 개수 | 32 또는 64 |
| `V` | vocabulary size: 선택 가능한 token 종류 수 | 13 |

`[B,T] = [2,5]`는 샘플 2개에 token이 각각 5개라는 뜻입니다. 축의 순서가 바뀐 `[T,B]`는 숫자 개수는 같아도 의미가 다릅니다. shape 오류가 나면 각 축의 의미와 순서에 유념하세요.


> **구분: 수업 후 복습·자습**

## 5. token → embedding → logits → 예측

1. 입력 token 번호의 shape는 `[B,T]`이고 자료형은 정수 `torch.long`입니다.
2. **Embedding** 층은 token 번호마다 학습 가능한 실수 벡터를 찾습니다. 출력은 `[B,T,H]`입니다. 사전처럼 번호로 행을 조회하지만, 각 행의 값은 훈련으로 바뀝니다.
3. 모델의 여러 층이 문맥을 반영한 hidden representation을 계산합니다.
4. 마지막 선형 층은 각 위치마다 vocabulary의 모든 후보에 대한 **logit** 을 냅니다. shape는 `[B,T,V]`입니다.
5. logit이 가장 큰 후보를 `argmax`로 고르면 예측 token 번호 `[B,T]`가 됩니다.

**logit** 은 확률이 되기 전의 제한 없는 실수 점수입니다. 음수여도 되고 합이 1일 필요도 없습니다. `softmax`를 적용하면 0–1 사이이고 합이 1인 확률 분포가 됩니다. 훈련의 `CrossEntropyLoss`에는 softmax를 미리 적용하지 않습니다. PyTorch 함수가 수치적으로 안정적인 방식으로 내부 처리하기 때문입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
embedding = torch.nn.Embedding(13, 8)
head = torch.nn.Linear(8, 13)
tokens = torch.tensor([[3, 4, 5]], dtype=torch.long)  # [B=1,T=3]
vectors = embedding(tokens)                           # [1,3,H=8]
logits = head(vectors)                                # [1,3,V=13]
probabilities = torch.softmax(logits, dim=-1)
predictions = logits.argmax(dim=-1)                   # [1,3]

print('tokens      :', tokens.shape, tokens.dtype)
print('embeddings  :', vectors.shape, vectors.dtype)
print('logits      :', logits.shape)
print('probability :', probabilities.shape,
      '한 위치의 합=', probabilities[0, 0].sum().item())
print('predictions :', predictions.shape, predictions)


> **구분: 수업 후 복습·자습**

## 6. loss, gradient, optimizer

<!-- procedure:lesson-11-0c296b:start -->
![한 번의 훈련과 검증](../교재그림/절차/lesson-11.svg)

평가 loss를 읽는 것과 모델 파라미터를 바꾸는 것을 구분합니다. shape와 정답 위치부터 확인하세요.
<!-- procedure:lesson-11-0c296b:end -->


**loss(손실)** 는 현재 예측이 정답과 얼마나 맞지 않는지를 하나의 숫자로 요약한 훈련용 기준입니다. 분류에서 많이 쓰는 cross entropy는 정답 token에 낮은 확률을 주면 큰 값을 냅니다. loss가 0에 가까울수록 훈련 목표에는 잘 맞지만, 특정 실험의 loss 숫자를 다른 데이터·모델의 loss와 그대로 비교해서는 안 됩니다.

**gradient(기울기)** 는 각 파라미터를 아주 조금 바꿨을 때 loss가 어느 방향으로 얼마나 변하는지를 나타냅니다. `loss.backward()`가 연쇄법칙으로 gradient를 계산해 각 파라미터의 `.grad`에 누적합니다. gradient 자체가 새로운 파라미터는 아닙니다.

**optimizer(최적화 알고리즘)** 는 gradient를 이용해 파라미터를 업데이트합니다. 이 실습은 Adam을 씁니다. **learning rate(학습률)** 는 한 번에 움직이는 크기를 조절합니다. 너무 크면 최솟값을 지나치거나 발산할 수 있고, 너무 작으면 훈련이 매우 느릴 수 있습니다.

훈련 한 step의 순서는 다음과 같습니다.

1. `optimizer.zero_grad(...)`: 이전 step에서 누적된 gradient를 비웁니다.
2. `logits = model(inputs)`: **forward pass** 로 예측 점수를 계산합니다.
3. `loss = criterion(logits, targets)`: 예측과 정답의 차이를 계산합니다.
4. `loss.backward()`: **backward pass(역전파)** 로 gradient를 계산합니다.
5. `optimizer.step()`: gradient를 보고 파라미터 값을 실제로 변경합니다.

**step** 은 batch 하나로 한 번 업데이트한 횟수입니다. **epoch** 는 훈련 데이터 전체를 한 번 사용한 단위입니다. 이 자료는 무작위 batch를 정해진 step 수만큼 뽑으므로 주로 step으로 훈련 길이를 표현합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
# 같은 정답에 높은 logit을 줄수록 cross entropy가 작아지는지 확인합니다.
targets_demo = torch.tensor([[2]])
wrong_logits = torch.tensor([[[4.0, 0.0, -2.0]]])
right_logits = torch.tensor([[[-2.0, 0.0, 4.0]]])
demo_loss = torch.nn.CrossEntropyLoss()
print('정답이 낮은 점수일 때 loss:', demo_loss(wrong_logits.reshape(-1, 3), targets_demo.reshape(-1)).item())
print('정답이 높은 점수일 때 loss:', demo_loss(right_logits.reshape(-1, 3), targets_demo.reshape(-1)).item())


> **구분: 수업 후 복습·자습**

## 7. 훈련 모드와 검증 모드

훈련에서는 파라미터를 업데이트하고, 검증에서는 고정된 파라미터로 성능만 측정합니다.

- `model.train()`: 모델을 훈련 모드로 둡니다. dropout, batch normalization처럼 모드에 따라 동작이 달라지는 층에 영향을 줍니다.
- `model.eval()`: 모델을 평가 모드로 둡니다. **gradient 계산을 자동으로 끄지는 않습니다.**
- `torch.no_grad()` 또는 `torch.inference_mode()`: gradient 기록을 꺼 메모리와 계산을 줄입니다.

따라서 검증에는 보통 `model.eval()`과 `no_grad()`가 모두 필요합니다. 검증 loss는 모델 선택과 과적합 관찰에 사용하지만, 검증 결과를 보고 같은 검증 집합에 끝없이 맞추면 검증 집합에도 간접적으로 과적합할 수 있습니다.


> **구분: 수업 후 복습·자습**

## 자습 구현 TODO — loss

수정 파일: **[04.LLM/exercises/loss.py](exercises/loss.py)**

logits `[B,T,V]`와 targets `[B,T]`를 CrossEntropyLoss 형태로 바꾸세요. softmax를 미리 적용하지 않습니다.
힌트: `reshape(-1, logits.shape[-1])`, `targets.reshape(-1)`, `F.cross_entropy`.
검증은 scalar loss 값뿐 아니라 gradient와 confidence 변화의 방향도 확인합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `token_loss(logits, targets)`에서 logits는 실수 텐서 `[B,T,V]`, targets는 정답 class 번호를 담은 정수 텐서 `[B,T]`입니다. targets는 같은 device에 있고 값은 `0..V-1`입니다. 반환은 평균 손실을 담은 0차원 텐서이며 역전파 연결을 유지합니다. `.item()`으로 숫자로 바꾸어 반환하지 않습니다.
**작성 흐름 힌트:** 위치 B×T개와 후보 V개를 구분 → 두 입력의 위치 순서를 같이 펼침 → raw logits와 정답 번호를 cross entropy에 전달. 모든 후보 점수가 같은 V=4 예에서는 손실이 약 `log(4)=1.3863`입니다. 이 수치를 고정 반환하는 문제가 아닙니다.
**설명할 지점:** “출력값 하나뿐 아니라 gradient가 모델까지 돌아가야 학습입니다. 먼저 softmax하거나 detach하면 어떤 차이가 생길까요?”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'exercises/loss.py').read_text())

In [ ]:
# 구분: 수업 후 복습·자습 셀
check('loss', 'exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습**

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/loss.py').read_text())

### 정답 해설 — 입력부터 결과까지

**두 tensor를 같은 순서로 펴기:** logits `[B,T,V]`에서 마지막 축 V는 후보 점수이므로 남기고 앞의 B·T를 합칩니다. targets `[B,T]`도 같은 순서로 펴야 각 점수 행과 정답 번호가 대응합니다. 예를 들어 `[2,3,4]`와 `[2,3]`은 `[6,4]`와 `[6]`입니다.

`F.cross_entropy`에는 확률이 아닌 raw logits와 정수 정답 index를 넣습니다. softmax를 미리 적용하지 않는 이유는 이 함수가 필요한 확률 계산을 내부에서 처리하기 때문입니다. 모든 후보 점수가 같고 후보가 4개라면 정답 확률은 1/4, loss는 약 1.386입니다. 정답 후보 점수가 상대적으로 커지면 loss가 줄어듭니다.

**결과 해석:** loss는 훈련에서 바꿀 방향을 계산할 기준이고, 실제로 파라미터를 갱신하는 것은 backward 뒤 optimizer.step입니다. 낮은 훈련 loss만으로 새로운 입력의 기억 성능을 입증하지 않습니다. 같은 검증 입력의 token/sequence 정확도와 훈련 전후 예측을 같이 봅니다.

**확인 질문:** logits만 다른 순서로 펼치면 shape가 맞아도 왜 엉뚱한 정답으로 학습하게 되나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/loss.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import torch.nn.functional as F

def token_loss(logits, targets):
    # CrossEntropyLoss는 softmax를 내부에서 처리하므로 raw logits을 전달합니다.
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('loss', 'solution')
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습**

### TODO의 shape 변환을 한 단계씩 읽기

`F.cross_entropy`의 일반적인 분류 입력은 `[N,C]`, 정답은 `[N]`입니다. 여기서 `C`는 class 수입니다. 우리 logits `[B,T,V]`에는 위치가 `B×T`개 있고, 각 위치마다 `V`개 후보가 있습니다. 따라서 다음처럼 해석합니다.

- logits: `[B,T,V] → [B×T,V]`
- targets: `[B,T] → [B×T]`

`reshape`는 위치들을 한 줄로 모을 뿐, 어느 logit이 어느 target과 짝인지 순서를 동일하게 유지합니다. `V` 축까지 섞어 `[B×T×V]`로 만들면 class 구조가 사라집니다. `view`도 가능하지만 메모리가 연속적이지 않은 텐서에서 실패할 수 있어 이 자료에서는 `reshape`를 사용합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
loss_op = load_op('loss')
model = torch.nn.Linear(3, 3).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
x = torch.eye(3, device='cuda').unsqueeze(0)
y = torch.tensor([[0,1,2]], device='cuda')
optimizer.zero_grad(set_to_none=True)
loss = loss_op.token_loss(model(x), y)
loss.backward()
print('loss:', loss.item(), 'gradient norm:', model.weight.grad.norm().item())
optimizer.step()
del model, optimizer, x, y, loss
torch.cuda.empty_cache()


> **구분: 수업 후 복습·자습**

## 8. 결과 읽기와 자주 생기는 오류

위 셀의 `gradient norm`이 0보다 크면 파라미터에 전달된 gradient가 있다는 뜻입니다. 하지만 한 step 뒤 loss가 항상 감소한다고 보장되지는 않습니다. 실제 판단은 여러 step의 추세와 별도 검증 지표를 함께 봅니다.

| 증상 | 흔한 원인 | 확인 방법 |
|---|---|---|
| `Expected ... Long` | token/target을 실수 dtype으로 만듦 | `tensor.dtype`이 `torch.long`인지 확인 |
| `Expected all tensors ... same device` | 일부는 CPU, 일부는 CUDA | 각 텐서와 모델 파라미터의 `.device` 확인 |
| size/shape mismatch | `[B,T,V]`와 `[B,T]` 축을 잘못 펼침 | loss 직전에 두 `.shape`를 출력 |
| gradient가 `None` | `backward()` 전이거나 계산 그래프에서 끊김 | `loss.requires_grad`, 파라미터 `.grad` 확인 |
| 두 번째 backward 오류 | 같은 계산 그래프를 다시 역전파함 | 새 forward를 수행; 무조건 `retain_graph=True`로 숨기지 않기 |
| CUDA 메모리 부족 | 큰 텐서/모델 참조가 남았거나 batch가 큼 | 사용 변수를 삭제하고 프로세스 종료 또는 커널 재시작 |
| loss가 `nan` | 너무 큰 학습률, 잘못된 연산, 발산 | 입력 범위·학습률·gradient norm부터 확인 |

`torch.cuda.empty_cache()`는 더 이상 참조되지 않는 캐시 블록을 allocator가 반환할 기회를 줍니다. 살아 있는 텐서를 삭제하지 않으며, 모델의 정확도나 학습 상태를 초기화하지도 않습니다.


> **구분: 수업 후 복습·자습**

## 완료·자습

loss 검증을 통과하고 backward와 step의 차이를 설명하세요. 자습(10분): 정답 token을 바꾸어 loss·gradient가 어떻게 달라지는지 비교합니다.
이후 각 모델은 새 프로세스에서 훈련합니다. GPU 메모리 정리와 재현 가능한 시작 상태를 확보하기 위한 구성입니다.
